In [4]:
"""
Run this where the full CSV currently lives (e.g. a Kaggle notebook).
It never loads the whole 800MB file into memory at once -- it streams
it in chunks. Two steps, controlled by the MODE flag below.

STEP 1: MODE = "inspect"
    Prints columns, dtypes, and per-subreddit comment counts.
    Use this to decide which subreddits to keep.

STEP 2: MODE = "sample"
    Filters to SUBREDDITS (if non-empty) and/or randomly downsamples,
    then writes a much smaller CSV you can upload to Claude.

Edit INPUT_PATH, MODE, SUBREDDITS, and SAMPLE_N as needed, then run.
"""

import pandas as pd

INPUT_PATH = "C:\\Users\\Hungthq\\MITB\\ISSS606_ADS_in_Social_Networks\\Group_Project\\GIT\\DATASETS\\cleaned_climate_comments_2025_05_to_2026_04.xls"  # <- set to actual filename
OUTPUT_PATH = "../PROCESSED/climate_reddit_filtered.csv"

MODE = "inspect"          # "inspect" first, then switch to "sample"
SUBREDDITS = []           # e.g. ["climate", "climatechange", "environment"] (lowercase)
SAMPLE_N = 150_000        # used only if SUBREDDITS is empty (random sample across all)
CHUNK_SIZE = 200_000

# ---------------------------------------------------------------- #

if MODE == "inspect":
    preview = pd.read_csv(INPUT_PATH, nrows=5)
    print("COLUMNS:", list(preview.columns))
    print(preview.dtypes)
    print("-" * 60)

    subreddit_col = next((c for c in preview.columns if "subreddit" in c.lower()), None)
    if subreddit_col is None:
        print("No column with 'subreddit' in its name was found.")
    else:
        counts = pd.Series(dtype="int64")
        for i, chunk in enumerate(pd.read_csv(INPUT_PATH, usecols=[subreddit_col],
                                                chunksize=CHUNK_SIZE, low_memory=False)):
            counts = counts.add(chunk[subreddit_col].value_counts(), fill_value=0)
            print(f"  scanned chunk {i + 1} ({(i + 1) * CHUNK_SIZE:,} rows so far)")
        print("-" * 60)
        print("Comment counts by subreddit:")
        print(counts.sort_values(ascending=False).astype(int).to_string())

elif MODE == "sample":
    kept_chunks = []
    total_rows = 0
    for i, chunk in enumerate(pd.read_csv(INPUT_PATH, chunksize=CHUNK_SIZE, low_memory=False)):
        subreddit_col = next((c for c in chunk.columns if "subreddit" in c.lower()), None)
        if SUBREDDITS and subreddit_col:
            chunk = chunk[chunk[subreddit_col].str.lower().isin(SUBREDDITS)]
        kept_chunks.append(chunk)
        total_rows += len(chunk)
        print(f"chunk {i + 1}: kept {len(chunk)} rows (running total {total_rows:,})")

    df = pd.concat(kept_chunks, ignore_index=True)

    if not SUBREDDITS and len(df) > SAMPLE_N:
        df = df.sample(n=SAMPLE_N, random_state=42)

    #df.to_csv(OUTPUT_PATH, index=False)
    df.to_csv("climate_reddit_filtered.csv", index=False)
    size_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"Saved {len(df):,} rows to {OUTPUT_PATH} (~{size_mb:.0f} MB in memory; "
          f"on-disk CSV will likely be smaller)")

else:
    raise ValueError("MODE must be 'inspect' or 'sample'")

COLUMNS: ['comment_id', 'score', 'self_text', 'subreddit', 'created_time', 'post_id', 'author_name', 'controversiality', 'ups', 'downs', 'user_is_verified', 'user_account_created_time', 'user_awardee_karma', 'user_awarder_karma', 'user_link_karma', 'user_comment_karma', 'user_total_karma', 'post_score', 'post_self_text', 'post_title', 'post_upvote_ratio', 'post_thumbs_ups', 'post_total_awards_received', 'post_created_time', 'comment_month', 'comment_date', 'comment_word_count', 'post_title_word_count', 'post_text_word_count']
comment_id                     object
score                           int64
self_text                      object
subreddit                      object
created_time                   object
post_id                        object
author_name                    object
controversiality                int64
ups                             int64
downs                           int64
user_is_verified                 bool
user_account_created_time      object
user_awardee